# Qwen3.5-0.8B LoRA Fine-tuning (v2)

Train Qwen on DialogSum with LoRA for length-controllable summarization.

**Run cells in order: 1 → 2 → 3 → 4 → 5.** Crash-safe: results saved to output directory.

### Hyperparameter changes vs v1 (previous Colab run)

| Parameter | v1 (old) | v2 (this) | Why |
|-----------|----------|-----------|-----|
| MAX_TARGET | 64 | **128** | LONG summaries were truncated |
| batch_size | 32 | **8** | LoRA needs finer gradient signal |
| learning_rate | 5e-5 | **2e-5** | Strong pre-trained model needs small nudge |
| epochs | 5 | **3** | Less overfitting risk |
| warmup | 100 steps | **ratio 0.1** | Proportional to total steps |
| num_beams (eval) | 1 | **4** | Beam search improves ROUGE |

### Saving A100 time: install on T4 first

1. **Runtime → T4 (free)** → run Cell 1 + Cell 2
2. **Runtime → switch to A100** → re-run Cell 1 (fast, kernels cached) + Cell 2 → 3 → 4 → 5

### Switching between local (500 samples) and Colab (full data)

Set `TRAIN_SAMPLES` in Cell 2:
- Local validation: `TRAIN_SAMPLES = 500`
- Colab full run: `TRAIN_SAMPLES = None` (use all 12,460)

In [ ]:
# @title Cell 1: Install dependencies
import subprocess, sys

!pip install -q --upgrade transformers
!pip install -q datasets peft rouge-score==0.1.2 tqdm accelerate

# CUDA kernel compilation — requires GPU runtime (T4 or A100)
# On CPU runtime this is safely skipped; re-run on GPU to compile.
import torch
HAS_CUDA = torch.cuda.is_available()

if HAS_CUDA:
    !pip install -q ninja packaging wheel setuptools
    print('=== Compiling causal-conv1d (5-10 min, please wait) ===')
    ret = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-v', '--no-build-isolation', 'causal-conv1d'],
        capture_output=True,
    )
    USE_FLASH_ATTN = False
    if ret.returncode == 0:
        print('=== Compiling flash-linear-attention ===')
        ret2 = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-v', '--no-build-isolation', 'flash-linear-attention'],
            capture_output=True,
        )
        USE_FLASH_ATTN = ret2.returncode == 0
    if USE_FLASH_ATTN:
        print('SUCCESS: Linear attention CUDA kernels installed!')
    else:
        print('WARNING: CUDA kernels not installed. Training works but linear attention will be slower (SDPA fallback).')
else:
    USE_FLASH_ATTN = False
    print('No CUDA detected — skipping kernel compilation.')
    print('If you plan to train on GPU, switch runtime to T4/A100 and re-run this cell.')

print(f'\nUSE_FLASH_ATTN = {USE_FLASH_ATTN}')
print('Ready.')

In [ ]:
# @title Cell 2: Initialize
import os, json, random, time, shutil
import numpy as np
from pathlib import Path

import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments,
    __version__ as transformers_version,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftConfig, PeftModel
from datasets import load_dataset
from tqdm import tqdm

# ── Configuration ──────────────────────────────────────────────────

SEED = 42
MODEL_NAME = 'Qwen/Qwen3.5-0.8B'
MAX_INPUT = 256
MAX_TARGET = 128           # was 64 — LONG summaries need more tokens

# Set to None for full dataset, or an int for local validation
TRAIN_SAMPLES = 500        # ← change to None for Colab full run
EVAL_SAMPLES = 500

SHORT_MAX, MEDIUM_MAX = 15, 35
BUCKET_RANGES = {'SHORT': (5, 15), 'MEDIUM': (16, 35), 'LONG': (36, 999)}
LENGTH_INSTRUCTIONS = {
    'SHORT': 'Write a very brief one-sentence summary of the dialogue in 5 to 15 words.',
    'MEDIUM': 'Write a short summary of the dialogue in 16 to 35 words.',
    'LONG': 'Write a detailed summary of the dialogue in more than 35 words.',
}

# Output directory
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    OUTPUT_BASE = Path('/content/drive/MyDrive/qwen_colab_v2')
else:
    OUTPUT_BASE = Path(__file__).resolve().parents[1] / 'results' / 'local_validation' if '__file__' in dir() else Path('results/local_validation')
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

# Reproducibility
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# Device
if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print(f'Device: cuda ({torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Device: MPS (Apple Silicon)')
else:
    device = torch.device('cpu')
    print('Device: CPU')

print(f'Transformers: {transformers_version}')

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'Tokenizer: {len(tokenizer)} tokens')

# Dataset
ds = load_dataset('knkarthick/dialogsum')
if TRAIN_SAMPLES:
    ds_train = ds['train'].select(range(TRAIN_SAMPLES))
    print(f'*** LOCAL VALIDATION: using {TRAIN_SAMPLES} train samples ***')
else:
    ds_train = ds['train']
    print(f'*** FULL DATASET: using {len(ds_train)} train samples ***')
ds_val = ds['validation']
ds_test = ds['test']
print(f'Train: {len(ds_train)}  Val: {len(ds_val)}  Test: {len(ds_test)}')
print('Init complete.')

In [ ]:
# @title Cell 3: Mount Google Drive (Colab only)
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Local mode: skipping Drive mount.')

In [ ]:
# @title Cell 4: Training & evaluation functions

def get_length_bucket(text):
    wc = len(text.split())
    return 'SHORT' if wc <= SHORT_MAX else 'MEDIUM' if wc <= MEDIUM_MAX else 'LONG'


def build_training_pairs(dlg, summ, topic, use_len, multi):
    pairs = []
    if use_len:
        instr = LENGTH_INSTRUCTIONS[get_length_bucket(summ)]
        prompt = (
            f'Summarize the following dialogue.\n'
            f'Instruction: {instr}\n'
            f'Dialogue:\n{dlg}\n'
            f'Summary: '
        )
    else:
        prompt = (
            f'Summarize the following dialogue.\n'
            f'Dialogue:\n{dlg}\n'
            f'Summary: '
        )
    pairs.append((prompt, summ))
    if multi:
        pairs.append((
            f'What is the topic of the following dialogue? Answer in a short phrase.\n'
            f'Dialogue:\n{dlg}\n'
            f'Topic: ',
            topic,
        ))
    return pairs


def preprocess_batch(batch, tok, use_length_tokens=False, multitask=False):
    ids, mask, labels = [], [], []
    mx = MAX_INPUT + MAX_TARGET
    for d, s, t in zip(batch['dialogue'], batch['summary'], batch['topic']):
        for p, tgt in build_training_pairs(d, s, t, use_length_tokens, multitask):
            pi = tok.encode(p, add_special_tokens=False)
            ti = tok.encode(tgt + tok.eos_token, add_special_tokens=False)
            if len(pi) + len(ti) > mx:
                pi = pi[-(mx - len(ti)):]
            fi = pi + ti
            ids.append(fi)
            mask.append([1] * len(fi))
            labels.append([-100] * len(pi) + list(ti))
    return {'input_ids': ids, 'attention_mask': mask, 'labels': labels}


class CausalLMCollator:
    def __init__(self, pad_token_id):
        self.pid = pad_token_id

    def __call__(self, features):
        ml = max(len(f['input_ids']) for f in features)
        batch = {'input_ids': [], 'attention_mask': [], 'labels': []}
        for f in features:
            p = ml - len(f['input_ids'])
            batch['input_ids'].append(f['input_ids'] + [self.pid] * p)
            batch['attention_mask'].append(f['attention_mask'] + [0] * p)
            batch['labels'].append(f['labels'] + [-100] * p)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}


def load_model():
    attn = 'flash_attention_2' if USE_FLASH_ATTN else 'sdpa'
    print(f'  Loading {MODEL_NAME} (attn={attn})...')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=torch.bfloat16, attn_implementation=attn,
    )
    lora_cfg = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=['q_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_cfg)
    model.gradient_checkpointing_disable()  # Incompatible with Qwen linear attention
    model.print_trainable_parameters()
    return model


def train_experiment(name, use_length_tokens=False, multitask=False, epochs=3, batch_size=8):
    od = OUTPUT_BASE / name
    od.mkdir(parents=True, exist_ok=True)

    if (od / 'training_complete.json').exists():
        print(f'SKIP {name}: already trained')
        return

    print(f"\n{'='*60}")
    print(f'  {name} | length_tokens={use_length_tokens} multitask={multitask} bs={batch_size} epochs={epochs}')
    print(f"{'='*60}")

    # Preprocess
    print('  Preprocessing...')
    kw = {'tok': tokenizer, 'use_length_tokens': use_length_tokens, 'multitask': multitask}
    tr = ds_train.map(preprocess_batch, fn_kwargs=kw, batched=True, batch_size=1000,
                      remove_columns=ds_train.column_names, desc='Train')
    va = ds_val.map(preprocess_batch, fn_kwargs=kw, batched=True, batch_size=500,
                    remove_columns=ds_val.column_names, desc='Val')
    print(f'  train={len(tr)} val={len(va)}')

    model = load_model()

    # Training arguments — key changes vs v1
    is_cuda = torch.cuda.is_available()
    args = TrainingArguments(
        output_dir=str(od),
        num_train_epochs=epochs,                # was 5
        per_device_train_batch_size=batch_size, # was 32
        per_device_eval_batch_size=batch_size,
        learning_rate=2e-5,                     # was 5e-5
        warmup_ratio=0.1,                       # was warmup_steps=100
        weight_decay=0.01,
        max_grad_norm=1.0,
        logging_steps=20,
        eval_strategy='steps',
        eval_steps=100,
        save_strategy='steps',
        save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        bf16=is_cuda,                           # bf16 on CUDA only
        gradient_checkpointing=False,           # Incompatible with Qwen linear attention
        dataloader_num_workers=4 if is_cuda else 0,
        dataloader_pin_memory=is_cuda,
        report_to='none',
        seed=SEED,
    )

    trainer = Trainer(
        model=model, args=args, train_dataset=tr, eval_dataset=va,
        processing_class=tokenizer,
        data_collator=CausalLMCollator(tokenizer.pad_token_id),
    )

    print('  Training...')
    t0 = time.time()
    result = trainer.train()
    elapsed = time.time() - t0

    trainer.save_model()
    tokenizer.save_pretrained(od)

    loss = float(result.metrics.get('train_loss', float('nan')))
    cfg = {
        'experiment': name, 'model': MODEL_NAME,
        'use_length_tokens': use_length_tokens, 'multitask': multitask,
        'train_samples': len(tr), 'epochs': epochs, 'batch_size': batch_size,
        'learning_rate': 2e-5, 'max_target': MAX_TARGET,
        'lora_modules': 6, 'lora_r': 16, 'lora_alpha': 32,
        'train_loss': loss, 'training_time_sec': round(elapsed, 1),
    }
    (od / 'config.json').write_text(json.dumps(cfg, indent=2))
    (od / 'training_complete.json').write_text(
        json.dumps({'status': 'done', 'loss': loss, 'elapsed': round(elapsed, 1)}, indent=2)
    )
    print(f'  Done. Loss={loss:.4f} Time={elapsed/60:.1f}min')

    # Clean checkpoints
    for c in sorted(od.glob('checkpoint-*')):
        shutil.rmtree(c, ignore_errors=True)


def evaluate_experiment(name, max_samples=500):
    md = OUTPUT_BASE / name
    if not (md / 'adapter_model.safetensors').exists():
        print(f'SKIP eval {name}: no model found')
        return None

    cfg = json.loads((md / 'config.json').read_text())
    use_len = cfg.get('use_length_tokens', False)

    print(f'  Loading adapter for {name}...')
    attn = 'flash_attention_2' if USE_FLASH_ATTN else 'sdpa'
    pc = PeftConfig.from_pretrained(str(md))
    base = AutoModelForCausalLM.from_pretrained(
        pc.base_model_name_or_path, dtype=torch.bfloat16, attn_implementation=attn,
    )
    model = PeftModel.from_pretrained(base, str(md)).to(device).eval()

    td = ds_test.select(range(min(max_samples, len(ds_test))))
    prompts, refs, buckets = [], [], []
    for r in td:
        refs.append(r['summary'])
        buckets.append(get_length_bucket(r['summary']))
        if use_len:
            inst = LENGTH_INSTRUCTIONS[get_length_bucket(r['summary'])]
            prompts.append(
                f'Summarize the following dialogue.\n'
                f'Instruction: {inst}\n'
                f'Dialogue:\n{r["dialogue"]}\n'
                f'Summary: '
            )
        else:
            prompts.append(
                f'Summarize the following dialogue.\n'
                f'Dialogue:\n{r["dialogue"]}\n'
                f'Summary: '
            )

    print(f'  Generating {len(prompts)} summaries...')
    preds = []
    eval_bs = 16 if torch.cuda.is_available() else 4
    for i in tqdm(range(0, len(prompts), eval_bs)):
        enc = tokenizer(
            prompts[i:i+eval_bs], max_length=MAX_INPUT, truncation=True,
            padding=True, return_tensors='pt',
        ).to(device)
        pl = enc['input_ids'].shape[1]
        with torch.no_grad():
            out = model.generate(
                **enc,
                max_new_tokens=MAX_TARGET,
                num_beams=4,                    # was 1 (greedy)
                early_stopping=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=False,
            )
        preds.extend(d.strip() for d in tokenizer.batch_decode(out[:, pl:], skip_special_tokens=True))

    # ROUGE
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    r1, r2, rL = [], [], []
    for p, ref in zip(preds, refs):
        s = scorer.score(ref, p)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rL.append(s['rougeL'].fmeasure)

    res = {
        'experiment': name, 'n': len(preds),
        'rouge1': round(sum(r1)/len(r1)*100, 2),
        'rouge2': round(sum(r2)/len(r2)*100, 2),
        'rougeL': round(sum(rL)/len(rL)*100, 2),
    }

    # Length accuracy
    if use_len:
        corr = sum(1 for p, b in zip(preds, buckets)
                   if BUCKET_RANGES[b][0] <= len(p.split()) <= BUCKET_RANGES[b][1])
        res['length_acc'] = round(corr / len(preds), 4)
        for bn in ['SHORT', 'MEDIUM', 'LONG']:
            ix = [i for i, b in enumerate(buckets) if b == bn]
            if ix:
                res[f'len_acc_{bn.lower()}'] = round(
                    sum(1 for i in ix
                        if BUCKET_RANGES[buckets[i]][0] <= len(preds[i].split()) <= BUCKET_RANGES[buckets[i]][1])
                    / len(ix), 4
                )

    (md / 'eval_results.json').write_text(json.dumps(res, indent=2))
    print(f'  R1={res["rouge1"]:.2f} R2={res["rouge2"]:.2f} RL={res["rougeL"]:.2f}', end='')
    if 'length_acc' in res:
        print(f'  LenAcc={res["length_acc"]*100:.1f}%', end='')
    print(f'\n  Saved to {md / "eval_results.json"}')
    return res


def run_experiment(name, use_len, multi, epochs=3, batch_size=8):
    ep = OUTPUT_BASE / name / 'eval_results.json'
    if ep.exists():
        r = json.loads(ep.read_text())
        print(f'SKIP {name}: RL={r["rougeL"]:.2f}')
        return r
    train_experiment(name, use_len, multi, epochs, batch_size)
    return evaluate_experiment(name)

print('Functions ready.')

## Cell 5: Run all experiments

Crash-safe: completed experiments are skipped.
If interrupted, re-run Cell 2 + Cell 4 + this cell to resume.

In [ ]:
# @title Cell 5: Run all
EXPS = [
    ('exp0_qwen_v2',        False, False),
    ('exp1_qwen_v2',        True,  False),
    ('exp1_multi_qwen_v2',  True,  True),
]

print(f"{'='*60}\nPipeline start\n{'='*60}")
for n, _, _ in EXPS:
    ep = OUTPUT_BASE / n / 'eval_results.json'
    status = 'done' if ep.exists() else 'new'
    print(f'  {n}: {status}')

for n, ul, mt in EXPS:
    print(f"\n{'='*60}")
    run_experiment(n, ul, mt, epochs=3, batch_size=8)

# Summary
print(f"\n{'='*70}\nALL RESULTS\n{'='*70}")
print(f"{'Experiment':<22} {'R1':>6} {'R2':>6} {'RL':>6} {'LenAcc':>8} {'S':>5} {'M':>5} {'L':>5}")
print('-' * 70)
for n, _, _ in EXPS:
    p = OUTPUT_BASE / n / 'eval_results.json'
    if p.exists():
        r = json.loads(p.read_text())
        la = f"{r['length_acc']*100:.1f}%" if 'length_acc' in r else '—'
        s = f"{r.get('len_acc_short', 0)*100:.0f}%" if 'len_acc_short' in r else '—'
        m = f"{r.get('len_acc_medium', 0)*100:.0f}%" if 'len_acc_medium' in r else '—'
        l = f"{r.get('len_acc_long', 0)*100:.0f}%" if 'len_acc_long' in r else '—'
        print(f"{n:<22} {r['rouge1']:>6.2f} {r['rouge2']:>6.2f} {r['rougeL']:>6.2f} {la:>8} {s:>5} {m:>5} {l:>5}")
    else:
        print(f'{n:<22} NOT DONE')

# Comparison with v1
print(f"\n{'='*70}\nvs V1 RESULTS\n{'='*70}")
v1 = {
    'exp0_qwen_v2':        [('300-sample baseline',  34.31), ('Full baseline (Colab)',  30.31)],
    'exp1_qwen_v2':        [('300-sample len ctrl',   33.41), ('Full len ctrl (Colab)',  30.08)],
    'exp1_multi_qwen_v2':  [('300-sample multi-task', 34.29), ('Full multi-task (Colab)', 29.41)],
}
for n, _, _ in EXPS:
    p = OUTPUT_BASE / n / 'eval_results.json'
    if not p.exists(): continue
    new_rl = json.loads(p.read_text())['rougeL']
    for label, prev_rl in v1.get(n, []):
        delta = new_rl - prev_rl
        sign = '+' if delta >= 0 else ''
        print(f'  {label:<30} {prev_rl:.2f} → {new_rl:.2f}  ({sign}{delta:.2f})')

In [ ]:
# @title Cell 6: Inspect sample predictions
import random as _r
_r.seed(0)

for n, _, _ in EXPS:
    md = OUTPUT_BASE / n
    ep = md / 'eval_results.json'
    if not ep.exists(): continue
    cfg = json.loads((md / 'config.json').read_text())
    use_len = cfg.get('use_length_tokens', False)

    # Re-generate a few samples
    pc = PeftConfig.from_pretrained(str(md))
    attn = 'flash_attention_2' if USE_FLASH_ATTN else 'sdpa'
    base = AutoModelForCausalLM.from_pretrained(pc.base_model_name_or_path, dtype=torch.bfloat16, attn_implementation=attn)
    model = PeftModel.from_pretrained(base, str(md)).to(device).eval()

    idx = _r.sample(range(len(ds_test)), 3)
    print(f"\n{'='*60}\n  {n} — sample predictions\n{'='*60}")
    for i in idx:
        r = ds_test[i]
        bucket = get_length_bucket(r['summary'])
        if use_len:
            inst = LENGTH_INSTRUCTIONS[bucket]
            prompt = f'Summarize the following dialogue.\nInstruction: {inst}\nDialogue:\n{r["dialogue"]}\nSummary: '
        else:
            prompt = f'Summarize the following dialogue.\nDialogue:\n{r["dialogue"]}\nSummary: '
        enc = tokenizer(prompt, max_length=MAX_INPUT, truncation=True, return_tensors='pt').to(device)
        pl = enc['input_ids'].shape[1]
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_TARGET, num_beams=4,
                                 pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        pred = tokenizer.decode(out[0, pl:], skip_special_tokens=True).strip()
        ref_wc = len(r['summary'].split())
        pred_wc = len(pred.split())
        print(f'\n  [{bucket}] ref={ref_wc}w pred={pred_wc}w')
        print(f'  REF: {r["summary"][:150]}')
        print(f'  GEN: {pred[:150]}')
    del model, base
    torch.cuda.empty_cache() if torch.cuda.is_available() else None